# EpistemicTrap — Analysis Report

This notebook summarizes benchmark results from exported `*.run.json` logs and produces a model-by-task score table with uncertainty estimates.


In [ ]:
import glob
import json
import math
import os
import random
from collections import defaultdict

random.seed(20260328)

def find_run_logs():
    paths = set()
    for pat in ['*.run.json', '**/*.run.json']:
        for p in glob.glob(pat, recursive=True):
            if os.path.isfile(p):
                paths.add(p)
    return sorted(paths)

def extract_numeric_results(run_json: dict):
    out = []
    for c in run_json.get('conversations', []) or []:
        r1 = c.get('result')
        if isinstance(r1, dict):
            nr = r1.get('numericResult')
            if isinstance(nr, dict) and nr.get('value') is not None:
                try:
                    out.append(float(nr.get('value')))
                    continue
                except Exception:
                    pass
        rs = c.get('results')
        if isinstance(rs, list):
            for r in rs:
                if not isinstance(r, dict):
                    continue
                nr = r.get('numericResult')
                if isinstance(nr, dict) and nr.get('value') is not None:
                    try:
                        out.append(float(nr.get('value')))
                    except Exception:
                        pass
    return out

def bootstrap_mean_ci(xs, iters=1000, alpha=0.05):
    if not xs:
        return None
    n = len(xs)
    means = []
    for _ in range(iters):
        sample = [xs[random.randrange(n)] for _ in range(n)]
        means.append(sum(sample) / n)
    means.sort()
    lo = means[int((alpha / 2) * iters)]
    hi = means[int((1 - alpha / 2) * iters) - 1]
    return (sum(xs) / n, lo, hi)

paths = find_run_logs()
print('Found run logs:', len(paths))
paths[:10]


In [ ]:
rows = []
def infer_task_name(run: dict, file_path: str) -> str:
    tv = run.get('taskVersion')
    if isinstance(tv, dict):
        name = tv.get('name')
        if isinstance(name, str) and name.strip():
            return name.strip()
    t = run.get('task')
    if isinstance(t, dict):
        name = t.get('name')
        if isinstance(name, str) and name.strip():
            return name.strip()
    name = run.get('taskName') or run.get('task_name')
    if isinstance(name, str) and name.strip():
        return name.strip()
    base = os.path.basename(file_path)
    return os.path.splitext(base)[0]

def infer_model_name(run: dict) -> str:
    m = run.get('model')
    if isinstance(m, dict):
        name = m.get('name')
        if isinstance(name, str) and name.strip():
            return name.strip()
    name = run.get('modelName') or run.get('model_name') or run.get('llmName') or run.get('llm_name')
    if isinstance(name, str) and name.strip():
        return name.strip()
    return 'unknown_model'

for p in paths:
    with open(p, 'r', encoding='utf-8') as f:
        run = json.load(f)
    vals = extract_numeric_results(run)
    if not vals:
        continue
    task = infer_task_name(run, p)
    model = infer_model_name(run)
    rows.append((task, model, vals))

print('Parsed runs:', len(rows))
rows[:3]


In [ ]:
by_task_model = defaultdict(list)
for task, model, vals in rows:
    by_task_model[(task, model)].extend(vals)

summary = []
for (task, model), vals in sorted(by_task_model.items()):
    ci = bootstrap_mean_ci(vals)
    if not ci:
        continue
    mean, lo, hi = ci
    summary.append((task, model, len(vals), mean, lo, hi))

summary[:10]


In [ ]:
def print_table(rows, headers):
    cols = list(zip(*([headers] + rows))) if rows else [headers]
    widths = [max(len(str(x)) for x in col) for col in cols]
    fmt = ' | '.join('{:' + str(w) + '}' for w in widths)
    print(fmt.format(*headers))
    print('-+-'.join('-' * w for w in widths))
    for r in rows:
        print(fmt.format(*r))

table = []
for task, model, n, mean, lo, hi in summary:
    table.append((task, model, str(n), f"{mean:.3f}", f"{lo:.3f}", f"{hi:.3f}"))

print_table(table, headers=('task','model','n','mean','ci_lo','ci_hi'))


## Pairwise Differences

This section computes bootstrap confidence intervals for mean score differences between models within each task.


In [ ]:
def bootstrap_diff_ci(a, b, iters=1000, alpha=0.05):
    if not a or not b:
        return None
    na = len(a)
    nb = len(b)
    diffs = []
    for _ in range(iters):
        sa = sum(a[random.randrange(na)] for _ in range(na)) / na
        sb = sum(b[random.randrange(nb)] for _ in range(nb)) / nb
        diffs.append(sa - sb)
    diffs.sort()
    mu = (sum(a) / na) - (sum(b) / nb)
    lo = diffs[int((alpha / 2) * iters)]
    hi = diffs[int((1 - alpha / 2) * iters) - 1]
    return (mu, lo, hi)

diff_table = []
tasks = sorted({t for (t, _) in by_task_model.keys()})
for task in tasks:
    models = sorted({m for (t, m) in by_task_model.keys() if t == task})
    for i in range(len(models)):
        for j in range(i + 1, len(models)):
            a = by_task_model[(task, models[i])]
            b = by_task_model[(task, models[j])]
            dci = bootstrap_diff_ci(a, b)
            if not dci:
                continue
            dmu, dlo, dhi = dci
            diff_table.append((task, models[i], models[j], f"{dmu:.3f}", f"{dlo:.3f}", f"{dhi:.3f}"))

print_table(diff_table, headers=('task','model_a','model_b','diff_mean','ci_lo','ci_hi'))


## Gradient Checks

Use this section to sanity-check that each task produces non-degenerate distributions (not all 0.0 or all 1.0) across models and across items.


In [ ]:
by_task = defaultdict(list)
for (task, model), vals in by_task_model.items():
    by_task[task].extend(vals)

degenerate = []
for task, vals in sorted(by_task.items()):
    mn = min(vals)
    mx = max(vals)
    mean = sum(vals) / len(vals)
    degenerate.append((task, len(vals), f"{mn:.3f}", f"{mx:.3f}", f"{mean:.3f}"))

print_table(degenerate, headers=('task','n','min','max','mean'))
